<a href="https://colab.research.google.com/github/viviantothewu-web/trashnet-waste-classification/blob/main/02_Shared_Training_Pipeline_Reformatted.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Disclosure

This "Reformatted" version is organized by ChatGPT. The original version "02_Shared_Training_Pipeline.ipynb" is included in Github and Google Docs as well, but this version is created so all teammembers can easily follow along without going through extra notes and texts in the original version.

# 02 — Shared PyTorch Training Pipeline

This notebook defines the **shared data and training pipeline** used for all four CNN architectures:

- Custom CNN
- VGG16
- ResNet50
- MobileNetV2

The goal is to make the comparison fair: every teammate and every architecture uses the same dataset split, image size, batch size, label mapping, augmentation policy, loss function, and training procedure.

> **Important:** This notebook is the shared reference pipeline. Do **not** add architecture-specific model code here. Model experiments will go into separate teammate experiment notebooks.

## Team Instructions

### Change only your assigned training seed

- **Vivian:** `TRAINING_SEED = 0`
- **Joshua:** `TRAINING_SEED = 1`
- **Devan:** `TRAINING_SEED = 2`

### Do not change these independently

- `RANDOM_STATE = 42`
- `IMG_SIZE = 224`
- `BATCH_SIZE = 32`
- class order
- fixed train / validation / test split
- training augmentation
- validation / test transformations
- shared loss and training functions

If the team decides to change one of these settings, everyone should use the same updated version.

### Before running in Colab

Go to **Runtime → Change runtime type → GPU**.

## Pipeline Overview

```text
TrashNet (2,527 images)
        ↓
Load fixed split from Notebook 01
        ↓
Train / Validation / Test DataFrames
        ↓
Class names → fixed numeric labels
        ↓
Resize + transforms + tensor conversion
        ↓
Custom PyTorch Dataset
        ↓
DataLoaders (batches of 32)
        ↓
Move each batch to GPU during training
        ↓
Shared train / validation loop
        ↓
Save best model weights
```

## 1. Environment and Imports

PyTorch will use a GPU when CUDA is available. The code automatically falls back to CPU if needed.

In [1]:

import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

PyTorch version: 2.11.0+cpu
CUDA available: False
Using device: cpu


## 2. Shared Experiment Settings

- `RANDOM_STATE = 42` was used in Notebook 01 to create the fixed 70/15/15 split.
- `TRAINING_SEED` changes by teammate so each architecture is trained under three different random conditions.
- Images are resized to **224 × 224** because this is a standard input size for VGG16, ResNet50, and MobileNetV2 and keeps the comparison consistent.
- `BATCH_SIZE = 32` is a practical balance between GPU memory use and training efficiency.

In [2]:
# Shared settings — keep these fixed across architectures
RANDOM_STATE = 42
TRAINING_SEED = 0   # Vivian = 0, Joshua = 1, Devan = 2
IMG_SIZE = 224
BATCH_SIZE = 32

CLASS_NAMES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

NUM_CLASSES = len(CLASS_NAMES)

print("Random State:", RANDOM_STATE)
print("Training seed:", TRAINING_SEED)
print("Image size:", IMG_SIZE)
print("Batch size:", BATCH_SIZE)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

Random State: 42
Training seed: 0
Image size: 224
Batch size: 32
Number of classes: 6
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


## 3. Reproducibility

Each teammate uses a different training seed. This changes training randomness while keeping the dataset split fixed.

In [3]:
# Set random seeds
random.seed(TRAINING_SEED)
np.random.seed(TRAINING_SEED)
torch.manual_seed(TRAINING_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(TRAINING_SEED)

print("Training seed set to:", TRAINING_SEED)

Training seed set to: 0


## 4. Load and Verify TrashNet

TrashNet is downloaded into the temporary Colab runtime. The dataset is checked to confirm that the expected **2,527 images** are present.

In [4]:
# Download TrashNet only if it is not already in the current runtime
if not Path("trashnet").exists():
    !git clone https://github.com/garythung/trashnet.git

DATASET_PATH = Path("trashnet/data/dataset-resized")

# Extract the image folders only if needed
if not DATASET_PATH.exists():
    !unzip -q trashnet/data/dataset-resized.zip -d trashnet/data/

print("Dataset available:", DATASET_PATH.exists())
print("Dataset path:", DATASET_PATH)

Cloning into 'trashnet'...
remote: Enumerating objects: 45, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 45 (delta 12), reused 6 (delta 6), pack-reused 27 (from 1)
Receiving objects: 100% (45/45), 40.64 MiB | 19.38 MiB/s, done.
Resolving deltas: 100% (14/14), done.
Dataset available: True
Dataset path: trashnet/data/dataset-resized


In [5]:
# Verify the expected number of images per class
total_images = 0

for class_name in CLASS_NAMES:
    class_path = DATASET_PATH / class_name
    image_count = len(list(class_path.glob("*.jpg")))
    print(f"{class_name}: {image_count}")
    total_images += image_count

print("\nTotal images:", total_images)
assert total_images == 2527, "Unexpected TrashNet image count. Check the dataset copy."

cardboard: 403
glass: 501
metal: 410
paper: 594
plastic: 482
trash: 137

Total images: 2527


## 5. Load the Fixed Train / Validation / Test Split

The split was created once in Notebook 01 and saved as `trashnet_split.csv`.

**Do not call `train_test_split()` again in this notebook.** Every teammate and every model must use the exact same image assignments.

If the CSV is not already in the Colab runtime, the next cell will ask you to upload it.

In [6]:
# Load the locked split created in Notebook 01
SPLIT_CSV = Path("trashnet_split.csv")

if not SPLIT_CSV.exists():
    from google.colab import files
    uploaded = files.upload()

split_df = pd.read_csv(SPLIT_CSV)

print("Total images in split:", len(split_df))
print("\nImages per split:")
print(split_df["split"].value_counts())

assert len(split_df) == 2527, "Split file should contain 2,527 rows."

Saving trashnet_split.csv to trashnet_split.csv
Total images in split: 2527

Images per split:
split
train         1768
test           380
validation     379
Name: count, dtype: int64


In [7]:
# Create the three fixed subsets — no new random splitting happens here
train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "validation"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Training images: 1768
Validation images: 379
Test images: 380
Total: 2527


## 6. Fixed Class Mapping

PyTorch trains on numeric labels rather than class names. This mapping must stay identical for every model and every teammate.

In [8]:
class_to_idx = {
    class_name: idx
    for idx, class_name in enumerate(CLASS_NAMES)
}

idx_to_class = {
    idx: class_name
    for class_name, idx in class_to_idx.items()
}

print("Class to index:", class_to_idx)
print("Index to class:", idx_to_class)

Class to index: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
Index to class: {0: 'cardboard', 1: 'glass', 2: 'metal', 3: 'paper', 4: 'plastic', 5: 'trash'}


## 7. Image Transformations

### Training images
Training images are resized to 224 × 224, then given small random horizontal flips and rotations to reduce overfitting and introduce limited visual variation.

### Validation and test images
Validation and test images are only resized and converted to tensors. They are **not randomly augmented** so evaluation remains consistent.

### Image Normalization

All images are normalized using the standard ImageNet mean and standard
deviation. This preprocessing matches the input expected by the pretrained
VGG16, ResNet50, and MobileNetV2 models and is kept consistent across the
team's experiments.

> Architecture-specific normalization for pretrained ImageNet models will be handled later at the model level.

In [9]:
# Validation / test transforms
# No augmentation is applied, but the same ImageNet normalization is used.

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Training transforms: same base preprocessing + limited augmentation
# Resize, apply light augmentation, convert to tensor,
# then normalize using ImageNet statistics.

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Training transform", train_transform)
print("Evaluation transform:", eval_transform)

Training transform Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)
Evaluation transform: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## 8. Custom PyTorch Dataset

`TrashNetDataset` connects each row in the fixed CSV split to the actual image file. For each sample it:

1. reads the filepath and class name,
2. opens the image as RGB,
3. converts the class name to the fixed numeric label,
4. applies the correct transform,
5. returns the processed image and label.

In [10]:
class TrashNetDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = row["filepath"]
        class_name = row["class"]

        image = Image.open(image_path).convert("RGB")
        label = class_to_idx[class_name]

        if self.transform:
            image = self.transform(image)

        return image, label

## 9. Create Datasets and DataLoaders

- Training uses augmentation and shuffling.
- Validation and test use deterministic transforms and are **not shuffled**, which helps keep predictions aligned with the original image records for later error analysis.

In [11]:
train_dataset = TrashNetDataset(train_df, transform=train_transform)
val_dataset = TrashNetDataset(val_df, transform=eval_transform)
test_dataset = TrashNetDataset(test_df, transform=eval_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))
print()
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training dataset: 1768
Validation dataset: 379
Test dataset: 380

Training batches: 56
Validation batches: 12
Test batches: 12


## 10. Pipeline Sanity Check

Before training any model, verify one batch has the expected shape and can be transferred to the selected device.

In [12]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

images = images.to(device)
labels = labels.to(device)

print("Images device:", images.device)
print("Labels device:", labels.device)

assert images.shape[1:] == (3, IMG_SIZE, IMG_SIZE)
assert labels.shape[0] == images.shape[0]

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Images device: cpu
Labels device: cpu


## Team Checkpoint Before Training

Confirm all of the following:

- [ ] Colab GPU is enabled.
- [ ] `CUDA available` is `True` when using GPU.
- [ ] TrashNet contains `2527` images.
- [ ] `trashnet_split.csv` contains `2527` rows.
- [ ] The fixed split loaded successfully.
- [ ] Class mapping has not changed.
- [ ] Image size is `224 × 224`.
- [ ] Batch size is `32`.
- [ ] Your assigned `TRAINING_SEED` is correct.
- [ ] A training batch has shape `[32, 3, 224, 224]` (except possibly the final smaller batch).
- [ ] Images and labels can move to the selected device.

# Shared Training Utilities

The remaining cells define the loss function and shared training / validation logic. These functions should be reused for all four architectures.

## 11. Loss Function

Cross-Entropy Loss is used because this is a single-label, six-class classification problem. It measures how far the model's predictions are from the correct class.

In [13]:
criterion = torch.nn.CrossEntropyLoss()
print("Loss function:", criterion)

Loss function: CrossEntropyLoss()


## 12. Train for One Epoch

One epoch is one complete pass through the training set. For each batch, the model predicts, calculates loss, backpropagates gradients, and updates its weights.

In [14]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predictions = torch.max(outputs, 1)
        correct_predictions += (predictions == labels).sum().item()
        total_images += labels.size(0)

    epoch_loss = running_loss / total_images
    epoch_accuracy = correct_predictions / total_images

    return epoch_loss, epoch_accuracy

## 13. Validate for One Epoch

Validation measures model performance without changing the model's weights. Gradients are disabled to save memory and computation.

In [15]:
def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predictions = torch.max(outputs, 1)
            correct_predictions += (predictions == labels).sum().item()
            total_images += labels.size(0)

    epoch_loss = running_loss / total_images
    epoch_accuracy = correct_predictions / total_images

    return epoch_loss, epoch_accuracy

## 14. Full Training Loop + Best-Model Checkpoint

This function runs training and validation for multiple epochs, stores the learning history, and saves the model weights whenever validation accuracy improves.

The **validation set** chooses the best epoch. The **test set is not used for model selection**.

In [16]:
def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    save_path,
):
    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    best_val_accuracy = 0.0

    for epoch in range(num_epochs):
        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device,
        )

        val_loss, val_accuracy = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device,
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), save_path)
            print(f"  Best model saved (Val Acc: {best_val_accuracy:.4f})")

    return history

## 15. Saved Model Folder and Naming Convention

The team will eventually create **12 trained model instances**: 4 architectures × 3 training seeds.

Use consistent filenames such as:

- `customcnn_seed0.pth`
- `vgg16_seed0.pth`
- `resnet50_seed0.pth`
- `mobilenetv2_seed0.pth`

Joshua and Devan will use the same pattern with seeds `1` and `2`.

In [17]:
MODEL_DIR = Path("saved_models")
MODEL_DIR.mkdir(exist_ok=True)

print("Model checkpoint folder:", MODEL_DIR)

Model checkpoint folder: saved_models


At this point the shared infrastructure is ready:

- fixed dataset split
- consistent labels
- image transforms
- PyTorch Dataset and DataLoaders
- GPU transfer
- loss function
- train / validation functions
- best-model checkpointing

**Do not add Custom CNN, VGG16, ResNet50, or MobileNetV2 architecture code to this notebook.**

The next step is to create separate experiment notebooks that import or reuse this shared pipeline and differ primarily by `TRAINING_SEED`.

## Export Shared Pipeline Code

The data-loading and training functions developed in this notebook will now
be stored as reusable Python modules.

This allows all team members to import the exact same implementation in their
individual experiment notebooks instead of copying or rewriting the pipeline.

Shared modules:

- `src/data_pipeline.py` — dataset, transforms, and DataLoaders
- `src/training.py` — training, validation, and checkpointing functions

In [18]:
# CREATE SHARED SOURCE-CODE FOLDER

from pathlib import Path

SRC_DIR = Path("src")
SRC_DIR.mkdir(exist_ok=True)

# Allows Python to import files from this folder as modules.
(SRC_DIR / "__init__.py").touch()

print("Shared source folder created:", SRC_DIR.exists())

Shared source folder created: True


In [19]:
%%writefile src/data_pipeline.py

# SHARED DATA PIPELINE
# Used by Vivian, Joshua, and Devan.
# Do not change independently between experiments.

import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


CLASS_NAMES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash"
]

CLASS_TO_IDX = {
    class_name: idx
    for idx, class_name in enumerate(CLASS_NAMES)
}

IDX_TO_CLASS = {
    idx: class_name
    for class_name, idx in CLASS_TO_IDX.items()
}


class TrashNetDataset(Dataset):
    """
    Connects the fixed CSV split to the actual TrashNet image files.
    """

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        image_path = row["filepath"]
        class_name = row["class"]

        image = Image.open(image_path).convert("RGB")
        label = CLASS_TO_IDX[class_name]

        if self.transform:
            image = self.transform(image)

        return image, label


def create_dataloaders(
    split_csv_path,
    img_size=224,
    batch_size=32,
    training_seed=0
):
    """
    Creates the shared training, validation, and test DataLoaders.
    """

    split_df = pd.read_csv(split_csv_path)

    train_df = split_df[split_df["split"] == "train"].copy()
    val_df = split_df[split_df["split"] == "validation"].copy()
    test_df = split_df[split_df["split"] == "test"].copy()

    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(10),
        transforms.ToTensor()
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    train_dataset = TrashNetDataset(
        train_df,
        transform=train_transform
    )

    val_dataset = TrashNetDataset(
        val_df,
        transform=eval_transform
    )

    test_dataset = TrashNetDataset(
        test_df,
        transform=eval_transform
    )

    generator = torch.Generator()
    generator.manual_seed(training_seed)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    return train_loader, val_loader, test_loader

Writing src/data_pipeline.py


In [20]:
%%writefile src/training.py

# SHARED TRAINING FUNCTIONS
# Used by Vivian, Joshua, and Devan.
# Do not change independently between experiments.

import torch


def train_one_epoch(model, train_loader, criterion, optimizer, device):
    """
    Train the model for one complete pass through the training dataset.
    """

    model.train()

    running_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        correct_predictions += (predictions == labels).sum().item()
        total_images += labels.size(0)

    epoch_loss = running_loss / total_images
    epoch_accuracy = correct_predictions / total_images

    return epoch_loss, epoch_accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    """
    Evaluate the model on the validation set without updating weights.
    """

    model.eval()

    running_loss = 0.0
    correct_predictions = 0
    total_images = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predictions = torch.max(outputs, 1)

            correct_predictions += (predictions == labels).sum().item()
            total_images += labels.size(0)

    epoch_loss = running_loss / total_images
    epoch_accuracy = correct_predictions / total_images

    return epoch_loss, epoch_accuracy


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    device,
    num_epochs,
    save_path
):
    """
    Train a model for multiple epochs and save the best weights
    based on validation accuracy.
    """

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    best_val_accuracy = 0.0

    for epoch in range(num_epochs):

        train_loss, train_accuracy = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_accuracy = validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.4f}"
        )

        if val_accuracy > best_val_accuracy:

            best_val_accuracy = val_accuracy

            torch.save(
                model.state_dict(),
                save_path
            )

            print(
                f"  Best model saved "
                f"(Val Acc: {best_val_accuracy:.4f})"
            )

    return history

Writing src/training.py


In [21]:
# test the shared module
# Confirm that the reusable data and training code can be imported
# successfully before teammates begin using it.

from src.data_pipeline import (
    CLASS_NAMES,
    CLASS_TO_IDX,
    IDX_TO_CLASS,
    create_dataloaders
)

from src.training import (
    train_one_epoch,
    validate_one_epoch,
    train_model
)

print("Shared modules imported successfully.")
print("Classes:", CLASS_NAMES)

Shared modules imported successfully.
Classes: ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [22]:
from google.colab import files

files.download("src/data_pipeline.py")
files.download("src/training.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>